# 1. Dot products: a weighted sum and a geometric comparison

Learn to calculate a dot product manually, implement a checked scalar result, interpret units, and distinguish magnitude-sensitive dot scores from cosine similarity. Prerequisite: [vector addition and multiplication](08_vector_addition_and_multiplication.ipynb). We use an invented two-product basket for a weighted sum and separate dimensionless vectors for geometry. No weights are learned from data in this lesson.

## 2. What problem does this solve?

A vector may describe how much of each item is present. Another vector may describe each item's price. Multiplying corresponding values gives each item's contribution; adding those contributions gives a total. A dot product performs exactly that operation.

Later, a linear prediction uses features and learned weights in a similar weighted sum. Dot products also compare vector directions, but a raw dot score depends on both direction and size. We will separate those interpretations so that a longer vector is not automatically described as more similar in every application.

## 3. Intuition and analogy

A receipt lists quantity times unit price for each product, followed by a sum. The product order must match in the quantity list and price list. A mathematically correct receipt with apple quantities paired to orange prices is still wrong.

For geometry, imagine two arrows from the same origin. Their dot product is positive when they point broadly together, zero when perpendicular, and negative when they point broadly against each other. Scaling an arrow can change the raw score without changing its direction. Dividing by both lengths removes that size effect when neither arrow has zero length.

## 4. Terminology

A **dot product** or real Euclidean **inner product** multiplies matching coordinates and sums them into one scalar. A **contribution** is one coordinate's product before summing. A **norm** here is Euclidean vector length. **Orthogonal** means perpendicular under this dot-product geometry. **Cosine similarity** is the dot product divided by the product of vector lengths.

An **intercept** is an added offset; it is not part of the dot product unless an extra constant feature and corresponding weight are explicitly included. A **feature order** is a semantic contract as well as a shape condition. `u * v` in NumPy returns coordinate products, whereas `np.dot(u,v)` sums those products for one-dimensional arrays.

## 5. Mathematical foundations

For real vectors $u,v$ of equal dimension $d$,

$$u^Tv=u\cdot v=\sum_{j=0}^{d-1}u_jv_j.$$

$j$ is a zero-based coordinate index; $u_j$ and $v_j$ are the corresponding scalar components. Both input vectors have NumPy shape $(d,)$; the output is one scalar. The superscript $T$ denotes transpose in column-vector mathematical notation; its full array meaning comes in the transpose lesson.

Euclidean length is $\lVert u\rVert_2=\sqrt{\sum_j u_j^2}$. For nonzero vectors,

$$\operatorname{cosine}(u,v)=\frac{u\cdot v}{\lVert u\rVert_2\lVert v\rVert_2}.$$

For dimensionless vectors, this is dimensionless and lies between minus one and one up to numerical rounding. In geometric coordinates, $u\cdot v=\lVert u\rVert_2\lVert v\rVert_2\cos\theta$, where $\theta$ is the angle between the arrows. A zero vector has no direction, so cosine similarity needs a separate zero-vector policy.

## 6. Hand-calculated example

A basket contains two apples and three oranges. Prices are four dollars per apple and five dollars per orange. The total is $2\times4+3\times5=8+15=23$ dollars. Quantity and price units cancel within each contribution to leave dollars. A two-dollar delivery fee produces 25 dollars after adding an intercept.

For dimensionless arrows $u=(3,0)$ and $v=(2,2)$, dot product is $3\times2+0\times2=6$. Lengths are 3 and $\sqrt8$. Cosine is $6/(3\sqrt8)=1/\sqrt2\approx0.7071$, corresponding to 45 degrees. Arrow $(0,2)$ is perpendicular to $u$ and has dot product zero. Arrow $(-2,0)$ has dot product minus six.

## 7. Implementation from scratch

Validate lengths before multiplying paired entries. Use a compensated library summation of the products, but make the products themselves explicit so the mechanism is visible. This helper accepts finite ordinary Python numbers and returns a scalar; it does not silently broadcast mismatched arrays.

In [ ]:
# Enable embedded figures in this fresh notebook kernel.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
import math

def checked_dot(left, right):
    first, second = list(left), list(right)
    if not first or len(first) != len(second):
        raise ValueError('Use nonempty vectors with matching coordinate counts.')
    if any(isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value)
           for value in first + second):
        raise ValueError('Use finite ordinary numerical coordinates.')
    contributions = [a * b for a, b in zip(first, second)]
    return math.fsum(contributions)

quantities, prices = [2.0, 3.0], [4.0, 5.0]
contributions = [quantity * price for quantity, price in zip(quantities, prices)]
total = checked_dot(quantities, prices)
assert contributions == [8.0, 15.0] and total == 23.0
assert total + 2 == 25.0
print('Contributions:', contributions, 'dollars; total:', total, 'dollars')

The input validation does not protect every extreme magnitude from product overflow. This is a bounded teaching helper, not a replacement for numerical analysis or a production monetary calculation with explicit decimal rounding rules.

## 8. Inspect signs and direction

In [ ]:
u_list = [3.0, 0.0]
v_list = [2.0, 2.0]
perpendicular = [0.0, 2.0]
opposite = [-2.0, 0.0]
assert checked_dot(u_list, v_list) == 6
assert checked_dot(u_list, perpendicular) == 0
assert checked_dot(u_list, opposite) == -6
u_length = math.sqrt(checked_dot(u_list, u_list))
v_length = math.sqrt(checked_dot(v_list, v_list))
manual_cosine = checked_dot(u_list, v_list) / (u_length * v_length)
assert math.isclose(manual_cosine, 1 / math.sqrt(2))
print('Dot scores: together=6, perpendicular=0, opposite=-6')
print('Cosine:', manual_cosine)

Do not call a zero dot product statistical independence. These are deterministic coordinate vectors, not a pair of random variables with a probability distribution. Orthogonality answers a geometric question under a specified inner product.

## 9. Library implementations

NumPy supplies `dot` and `linalg.norm`. scikit-learn supplies cosine similarity for tables of vectors. A single vector must be made into a one-row table with shape `(1,2)` for that interface; its output is a `(1,1)` similarity table. The mathematical quantity remains one comparison.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
u, v = np.array(u_list), np.array(v_list)
products = u * v
scalar_dot = np.dot(u, v)
assert products.shape == (2,) and np.ndim(scalar_dot) == 0
np.testing.assert_array_equal(products, [6.0, 0.0])
assert scalar_dot == checked_dot(u_list, v_list)
library_cosine = cosine_similarity(u.reshape(1, -1), v.reshape(1, -1))
assert library_cosine.shape == (1, 1)
assert math.isclose(library_cosine[0, 0], manual_cosine)
assert math.isclose(np.linalg.norm(v), v_length)
print('Elementwise products:', products, 'Scalar dot:', scalar_dot)

The cosine library uses a convention for zero rows, but a meaningful direction is still absent. A scientific interpretation should disclose that policy instead of assuming a returned number creates a direction.

## 10. Visualization

The contribution bars show why the basket total is 23 dollars. The geometric panel is separate and dimensionless; it does not plot product quantities against dollar prices as if they were spatial arrows. Equal geometric axis scales preserve the visual meaning of perpendicular directions.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
bars = axes[0].bar(['Apples', 'Oranges'], contributions)
axes[0].bar_label(bars)
axes[0].set(ylabel='Contribution (dollars)', title='Coordinate products sum to 23 dollars', ylim=(0, 18))
for vector, label, color in [(u, 'u: right', 'steelblue'), (v, 'v: diagonal', 'darkorange'),
                              (np.array(perpendicular), 'Perpendicular', 'black')]:
    axes[1].quiver(0, 0, *vector, angles='xy', scale_units='xy', scale=1, label=label, color=color)
axes[1].set(xlabel='Coordinate 0 (dimensionless)', ylabel='Coordinate 1 (dimensionless)',
            xlim=(-0.5, 4), ylim=(-0.5, 4), aspect='equal', title='Direction and magnitude both affect dot')
axes[1].legend(loc='upper right', fontsize=8)
fig.tight_layout()
display(fig)
plt.close(fig)

## 11. Experiment: scale without changing direction

Double the diagonal arrow. Its dot product with $u$ doubles, but cosine stays the same. This explains why a raw dot score and normalized similarity are different measures. Neither is universally better: vector magnitude may contain meaningful information or may be an unwanted scale effect.

In [ ]:
twice_v = 2 * v
assert np.dot(u, twice_v) == 2 * np.dot(u, v) == 12
scaled_cosine = cosine_similarity(u.reshape(1, -1), twice_v.reshape(1, -1))[0, 0]
assert math.isclose(scaled_cosine, manual_cosine)
changed_order_total = checked_dot(quantities, list(reversed(prices)))
assert changed_order_total == 22 and changed_order_total != total
print('Scaled dot:', np.dot(u, twice_v), 'Unchanged cosine:', scaled_cosine)
print('Wrong feature order changes the basket total to:', changed_order_total)

## 12. Weights and similarity choices

Our prices and vectors are supplied inputs, not learned model parameters. In a later linear predictor, weights are learned and a separate intercept may be added. Choosing raw dot or cosine is a representation/evaluation choice. Scaling features or normalizing vectors changes the meaning of similarity. Arbitrary mixing of units can make a geometric angle difficult to interpret even when code returns a number.

## 13. Evaluation

Check contributions, final scalar shape, coordinate order, nonmutation, symmetry, and distributivity. For real vectors, $u\cdot v=v\cdot u$. The dot product also distributes over vector addition. Cosine requires nonzero lengths and a stated feature representation. Those mathematical identities are independent of any prediction-performance claim.

In [ ]:
np.testing.assert_allclose(np.dot(u, v), np.dot(v, u))
np.testing.assert_allclose(np.dot(u, v + u), np.dot(u, v) + np.dot(u, u))
assert checked_dot([0.0, 0.0], [1.0, 2.0]) == 0
for left, right in [([], []), ([1, 2], [3]), ([True], [2]), ([math.inf], [1])]:
    try:
        checked_dot(left, right)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid dot inputs were accepted.')
assert quantities == [2.0, 3.0] and prices == [4.0, 5.0]

## 14. Assumptions and limitations

Both vectors refer to matching coordinates in the same order. Geometric interpretations assume a Euclidean representation whose coordinate scales are meaningful. Mixed feature units or sparse high-dimensional representations require deliberate preprocessing and interpretation. The scratch helper handles real numbers only; complex inner products use conjugation and are outside its contract. No learned embeddings or real retrieval system are evaluated here.

## 15. Common mistakes and debugging

Do not stop at `u*v` when you intend a scalar weighted sum. Do not flatten a table indiscriminately to make a dot operation accept it. Check shape and coordinate names first. Do not normalize a zero vector by dividing by zero. Do not assume a larger raw dot score proves a closer direction. When a price calculation disagrees, inspect each product contribution before changing the total formula.

## 16. Alternatives and connections

Elementwise multiplication preserves separate contributions, a dot product aggregates them, and matrix multiplication performs many dot products at once. Euclidean distance measures separation; cosine compares direction after normalization. A weighted sum is useful for specified prices or linear predictions, but it does not automatically capture interactions between features. Later model families introduce those interactions deliberately.

## 17. Exercises

- **Beginner:** Compute the dot product of `(1,2,3)` and `(4,0,-1)`.
- **Beginner:** Find each contribution and total for quantities `(1,4)` and unit prices `(3,2)` dollars.
- **Beginner:** Explain the output-shape difference between an elementwise product and a dot product.
- **Intermediate:** Give a nonzero vector perpendicular to `(2,1)` and verify its dot product.
- **Intermediate:** Explain what multiplying one vector by a positive factor does to dot and cosine scores, and what a negative factor changes.
- **Challenge:** Implement a checked cosine helper using the scratch dot routine, reject zero vectors explicitly, and test identical, opposite, perpendicular, and unequal-length inputs.

## 18. Detailed solutions

The first dot is $1\times4+2\times0+3\times(-1)=1$. Basket contributions are three and eight dollars, totaling eleven. Elementwise multiplication returns three coordinates in the first example, while dot returns one scalar. Vector `(1,-2)` is perpendicular to `(2,1)` because $2-2=0$. Positive scaling multiplies a raw dot score but preserves cosine; negative scaling reverses direction and flips the cosine sign.

The challenge should use the validated dot routine first, then inspect both self-dots for zero lengths before division. It returns a meaningful angle-based quantity only for nonzero vectors. Floating-point roundoff can slightly exceed the theoretical cosine bounds in extreme cases; a later inverse-cosine conversion may need an explicit rounding policy.

In [ ]:
def checked_cosine(left, right):
    first, second = list(left), list(right)
    numerator = checked_dot(first, second)
    first_squared = checked_dot(first, first)
    second_squared = checked_dot(second, second)
    if first_squared == 0 or second_squared == 0:
        raise ValueError('Zero vectors have no defined direction.')
    return numerator / (math.sqrt(first_squared) * math.sqrt(second_squared))

assert checked_dot([1, 2, 3], [4, 0, -1]) == 1
assert checked_dot([1, 4], [3, 2]) == 11
assert checked_dot([2, 1], [1, -2]) == 0
assert math.isclose(checked_cosine([1, 2], [2, 4]), 1.0)
assert math.isclose(checked_cosine([1, 2], [-2, -4]), -1.0)
assert checked_cosine([2, 1], [1, -2]) == 0
for left, right in [([0, 0], [1, 2]), ([1, 2], [1])]:
    try:
        checked_cosine(left, right)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid cosine inputs were accepted.')
print('Weighted sums and checked cosine exercises passed.')

## 19. Knowledge check

**What does a dot product return?** One scalar from a sum of matching coordinate products.

**Why preserve coordinate order?** Each product must pair the intended quantity with its matching weight or coordinate.

**What does zero dot mean geometrically?** Perpendicular directions for two nonzero vectors under this Euclidean inner product.

**Why normalize for cosine?** To remove the magnitude factor and compare directions.

**Is an intercept part of the dot automatically?** No. It is added separately unless encoded as a constant feature.

## 20. Interview preparation

**How does a linear prediction use a dot product?** It sums feature-weight contributions and may add an intercept, with weights obtained by a separate training process.

**Why can dot-product similarity favor long vectors?** Magnitudes multiply the directional cosine, so scale affects the raw score.

**How do you handle zero-vector cosine?** State a policy explicitly; the direction is mathematically undefined even if a library chooses a numerical convention.

**Does orthogonality imply statistical independence?** No. A deterministic geometric relationship does not establish a probabilistic independence claim.

**How would you debug an incorrect weighted sum?** Check shape, coordinate labels, units, individual products, and the added offset separately.

## 21. Summary and next steps

A dot product aggregates coordinate products into a scalar. Its units and meaning depend on the inputs; cosine adds a nonzero-length normalization. Continue to [matrix multiplication](10_matrix_multiplication.ipynb), which arranges many dot products into a prediction table.